In [137]:
import pandas as pd 
import numpy as np

In [138]:
df = pd.read_csv("/Users/Banjo/OneDrive/Desktop/MIDS/Summer 2025/DataSci 207/Final Project/w207FinalProject/data/raw/plays.csv")
df

,gameId,playId,playDescription,quarter,down,yardsToGo,possessionTeam,defensiveTeam,yardlineSide,yardlineNumber,...,yardsGained,homeTeamWinProbabilityAdded,visitorTeamWinProbilityAdded,expectedPointsAdded,isDropback,pff_runConceptPrimary,pff_runConceptSecondary,pff_runPassOption,pff_passCoverage,pff_manZone
0,2022102302,2655,(1:54) (Shotgun) J.Burrow pass short middle to...,3,1,10,CIN,ATL,CIN,21,...,9,0.004634,-0.004634,0.702717,True,NaN,NaN,0,Cover-3,Zone
1,2022091809,3698,(2:13) (Shotgun) J.Burrow pass short right to ...,4,1,10,CIN,DAL,CIN,8,...,4,0.002847,-0.002847,-0.240509,True,NaN,NaN,0,Quarters,Zone
2,2022103004,3146,(2:00) (Shotgun) D.Mills pass short right to D...,4,3,12,HOU,TEN,HOU,20,...,6,0.000205,-0.000205,-0.218480,True,NaN,NaN,0,Quarters,Zone
3,2022110610,348,(9:28) (Shotgun) P.Mahomes pass short left to ...,1,2,10,KC,TEN,TEN,23,...,4,-0.001308,0.001308,-0.427749,True,NaN,NaN,0,Quarters,Zone
4,2022102700,2799,(2:16) (Shotgun) L.Jackson up the middle to TB...,3,2,8,BAL,TB,TB,27,...,-1,0.027141,-0.027141,-0.638912,False,MAN,READ OPTION,0,Cover-1,Man
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2022110604,1051,(12:49) (Shotgun) T.Lawrence pass incomplete s...,2,3,4,JAX,LV,JAX,31,...,0,-0.024741,0.024741,-1.391687,True,NaN,NaN,0,Cover-2,Zone
16120,2022103005,3492,(12:32) (Shotgun) K.Cousins pass incomplete de...,4,1,10,MIN,ARI,MIN,25,...,0,-0.026580,0.026580,-0.503208,True,NaN,NaN,0,Cover-3,Zone
16121,2022092502,2337,(9:59) (Shotgun) P.Mahomes scrambles right end...,3,1,10,KC,IND,IND,13,...,10,-0.013790,0.013790,1.073898,True,NaN,NaN,0,Quarters,Zone
16122,2022091809,719,(:45) C.Rush pass incomplete deep right to C.L...,1,1,10,DAL,CIN,CIN,47,...,0,-0.011561,0.011561,-0.522397,True,UNDEFINED,NaN,0,Cover-3,Zone


In [139]:
cols_to_drop = [
    "dropbackDistance", "expectedPoints", "expectedPointsAdded",
    "gameId", "homeTeamWinProbabilityAdded", "isDropback",
    "passLength", "passLocationType", "passResult", "passTippedAtLine",
    "penaltyYards", "pff_runConceptSecondary", "playClockAtSnap",
    "playDescription", "playId", "playNullifiedByPenalty", 
    "prePenaltyYardsGained", "preSnapHomeScore", "preSnapHomeTeamWinProbability",
    "preSnapVisitorScore", "preSnapVisitorTeamWinProbability", "qbKneel",
    "qbSpike", "targetX", "targetY", "timeInTackleBox",
    "timeToSack", "timeToThrow", "unblockedPressure",
    "visitorTeamWinProbilityAdded", "yardlineNumber", "yardlineSide"
]

df.drop(columns=cols_to_drop, errors="ignore", inplace=True)
df

,quarter,down,yardsToGo,possessionTeam,defensiveTeam,gameClock,absoluteYardlineNumber,offenseFormation,receiverAlignment,playAction,dropbackType,qbSneak,rushLocationType,yardsGained,pff_runConceptPrimary,pff_runPassOption,pff_passCoverage,pff_manZone
0,3,1,10,CIN,ATL,01:54,31,EMPTY,3x2,False,TRADITIONAL,NaN,NaN,9,NaN,0,Cover-3,Zone
1,4,1,10,CIN,DAL,02:13,18,EMPTY,3x2,False,TRADITIONAL,NaN,NaN,4,NaN,0,Quarters,Zone
2,4,3,12,HOU,TEN,02:00,30,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,6,NaN,0,Quarters,Zone
3,1,2,10,KC,TEN,09:28,33,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,4,NaN,0,Quarters,Zone
4,3,2,8,BAL,TB,02:16,37,PISTOL,3x1,True,DESIGNED_RUN,False,INSIDE_LEFT,-1,MAN,0,Cover-1,Man
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2,3,4,JAX,LV,12:49,79,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,0,NaN,0,Cover-2,Zone
16120,4,1,10,MIN,ARI,12:32,35,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,0,NaN,0,Cover-3,Zone
16121,3,1,10,KC,IND,09:59,97,SHOTGUN,2x2,False,SCRAMBLE,False,INSIDE_RIGHT,10,NaN,0,Quarters,Zone
16122,1,1,10,DAL,CIN,00:45,57,SINGLEBACK,2x2,True,TRADITIONAL,NaN,NaN,0,UNDEFINED,0,Cover-3,Zone


In [140]:
man_keywords = ['manunder', 'cover-1', 'man']
zone_keywords = ['cover-2', 'cover-3', 'cover-4', 'cover-6', 'quarters', 'cloud', 'zone']

def simplify_coverage(cov):
    if pd.isna(cov):
        return 'Other'
    v = str(cov).lower()
    # map to Man first to avoid accidental overlap
    for kw in man_keywords:
        if kw in v:
            return 'Man'
    for kw in zone_keywords:
        if kw in v:
            return 'Zone'
    # if it contains the word blitz assign Blitz, otherwise default to Blitz
    if 'blitz' in v:
        return 'Blitz'
    return 'Blitz'

# apply the mapping
df['pff_coverage'] = df['pff_passCoverage'].apply(simplify_coverage)

# inspect distributions
print("Simplified coverage value counts:")
print(df['pff_coverage'].value_counts(dropna=False))

print("\nTop original pff_passCoverage values:")
print(df['pff_passCoverage'].value_counts(dropna=False).head(20))
pff_passCoverage = ['pff_passCoverage']
df.drop(columns=pff_passCoverage)


Simplified coverage value counts:
pff_coverage
Zone     10814
Man       3540
Blitz     1578
Other      192
Name: count, dtype: int64

Top original pff_passCoverage values:
pff_passCoverage
Cover-3                 4956
Cover-1                 3300
Quarters                2073
Cover-2                 1852
Cover 6-Left             692
Cover-6 Right            690
Cover-3 Seam             636
Cover-0                  605
Red Zone                 537
NaN                      192
2-Man                    186
Goal Line                146
Bracket                   75
Cover-1 Double            54
Prevent                   46
Cover-3 Cloud Right       31
Cover-3 Cloud Left        30
Miscellaneous             14
Cover-3 Double Cloud       9
Name: count, dtype: int64


,quarter,down,yardsToGo,possessionTeam,defensiveTeam,gameClock,absoluteYardlineNumber,offenseFormation,receiverAlignment,playAction,dropbackType,qbSneak,rushLocationType,yardsGained,pff_runConceptPrimary,pff_runPassOption,pff_manZone,pff_coverage
0,3,1,10,CIN,ATL,01:54,31,EMPTY,3x2,False,TRADITIONAL,NaN,NaN,9,NaN,0,Zone,Zone
1,4,1,10,CIN,DAL,02:13,18,EMPTY,3x2,False,TRADITIONAL,NaN,NaN,4,NaN,0,Zone,Zone
2,4,3,12,HOU,TEN,02:00,30,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,6,NaN,0,Zone,Zone
3,1,2,10,KC,TEN,09:28,33,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,4,NaN,0,Zone,Zone
4,3,2,8,BAL,TB,02:16,37,PISTOL,3x1,True,DESIGNED_RUN,False,INSIDE_LEFT,-1,MAN,0,Man,Man
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2,3,4,JAX,LV,12:49,79,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,0,NaN,0,Zone,Zone
16120,4,1,10,MIN,ARI,12:32,35,SHOTGUN,2x2,False,TRADITIONAL,NaN,NaN,0,NaN,0,Zone,Zone
16121,3,1,10,KC,IND,09:59,97,SHOTGUN,2x2,False,SCRAMBLE,False,INSIDE_RIGHT,10,NaN,0,Zone,Zone
16122,1,1,10,DAL,CIN,00:45,57,SINGLEBACK,2x2,True,TRADITIONAL,NaN,NaN,0,UNDEFINED,0,Zone,Zone


In [141]:
# simplify dropbackType
def simplify_dropback(dt):
    if pd.isna(dt):
        return 'Other'
    v = str(dt).lower()
    if 'traditional' in v:
        return 'Traditional'
    if 'rollout' in v:
        return 'Rollout'
    if 'sneak' in v:
        return 'QB_Sneak'
    if 'run' in v:
        return 'Run'
    if 'scramble' in v:
        return 'Scramble'

    return 'Other'

# simplify offenseFormation
def simplify_formation(of):
    if pd.isna(of):
        return 'Other'
    v = str(of).lower()
    if 'empty' in v:
        return 'Empty'
    if 'shotgun' in v:
        return 'Shotgun'
    if 'pistol' in v:
        return 'Pistol'
    if 'singleback' in v or 'single back' in v:
        return 'Singleback'
    if 'i_form' in v or 'i-form' in v or v.startswith('i'):
        return 'I-Form'
    return 'Other'

df['dropbackType'] = df['dropbackType'].apply(simplify_dropback)
df['offenseFormation'] = df['offenseFormation'].apply(simplify_formation)

# inspect
print("dropback_type value counts:\n", df['dropbackType'].value_counts())
print("offenseFormation value counts:\n", df['offenseFormation'].value_counts())
df


dropback_type value counts:
 dropbackType
Traditional    8149
Other          5964
Scramble        870
Rollout         729
Run             301
QB_Sneak        111
Name: count, dtype: int64
offenseFormation value counts:
 offenseFormation
Shotgun       8791
Singleback    3915
Empty         1342
I-Form        1035
Pistol         641
Other          400
Name: count, dtype: int64


,quarter,down,yardsToGo,possessionTeam,defensiveTeam,gameClock,absoluteYardlineNumber,offenseFormation,receiverAlignment,playAction,dropbackType,qbSneak,rushLocationType,yardsGained,pff_runConceptPrimary,pff_runPassOption,pff_passCoverage,pff_manZone,pff_coverage
0,3,1,10,CIN,ATL,01:54,31,Empty,3x2,False,Traditional,NaN,NaN,9,NaN,0,Cover-3,Zone,Zone
1,4,1,10,CIN,DAL,02:13,18,Empty,3x2,False,Traditional,NaN,NaN,4,NaN,0,Quarters,Zone,Zone
2,4,3,12,HOU,TEN,02:00,30,Shotgun,2x2,False,Traditional,NaN,NaN,6,NaN,0,Quarters,Zone,Zone
3,1,2,10,KC,TEN,09:28,33,Shotgun,2x2,False,Traditional,NaN,NaN,4,NaN,0,Quarters,Zone,Zone
4,3,2,8,BAL,TB,02:16,37,Pistol,3x1,True,Run,False,INSIDE_LEFT,-1,MAN,0,Cover-1,Man,Man
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2,3,4,JAX,LV,12:49,79,Shotgun,2x2,False,Traditional,NaN,NaN,0,NaN,0,Cover-2,Zone,Zone
16120,4,1,10,MIN,ARI,12:32,35,Shotgun,2x2,False,Traditional,NaN,NaN,0,NaN,0,Cover-3,Zone,Zone
16121,3,1,10,KC,IND,09:59,97,Shotgun,2x2,False,Scramble,False,INSIDE_RIGHT,10,NaN,0,Quarters,Zone,Zone
16122,1,1,10,DAL,CIN,00:45,57,Singleback,2x2,True,Traditional,NaN,NaN,0,UNDEFINED,0,Cover-3,Zone,Zone


In [142]:
def extract_left_right(align):
    if isinstance(align, str) and "x" in align:
        parts = align.lower().split("x")
        try:
            left = int(parts[0]) if parts[0].isdigit() else 0
            right = int(parts[1]) if len(parts) > 1 and parts[1].isdigit() else 0
        except Exception:
            return 0, 0
        return left, right
    return 0, 0

def parse_game_clock(gc):
    if isinstance(gc, str) and ":" in gc:
        try:
            minutes, seconds = gc.split(":")
            return int(minutes) * 60 + int(seconds)
        except Exception:
            return pd.NA
    return pd.NA


# compute left/right receivers
lr = df["receiverAlignment"].apply(lambda x: extract_left_right(x))
df["leftReceivers"] = lr.apply(lambda t: t[0])
df["rightReceivers"] = lr.apply(lambda t: t[1])

# compute seconds remaining in quarter from gameClock (MM:SS)
df["secondsRemainingInQuarter"] = df["gameClock"].apply(parse_game_clock)
gameClock = ['gameClock', 'receiverAlignment', "rushLocationType", 'pff_runConceptPrimary', 'pff_runPassOption']
df = df.drop(columns=gameClock)

In [143]:
def add_team_tendency_features(
    df,
    offense_col="possessionTeam",
    defense_col="defensiveTeam",
    target="yardsGained",
    rolling_window=None,  # int for last N plays, or None for full expanding history
    min_periods=1,
    eps=1e-6,
    fill_na_with_global=True
):
    df = df.copy()
    # offensive / defensive tendency (prior to current play)
    if rolling_window is None:
        df["offense_mean_yards"] = (
            df.groupby(offense_col, group_keys=False)[target]
              .apply(lambda s: s.shift().expanding(min_periods=min_periods).mean())
        )
        df["defense_mean_allowed"] = (
            df.groupby(defense_col, group_keys=False)[target]
              .apply(lambda s: s.shift().expanding(min_periods=min_periods).mean())
        )
    else:
        df["offense_mean_yards"] = (
            df.groupby(offense_col, group_keys=False)[target]
              .apply(lambda s: s.shift().rolling(window=rolling_window, min_periods=min_periods).mean())
        )
        df["defense_mean_allowed"] = (
            df.groupby(defense_col, group_keys=False)[target]
              .apply(lambda s: s.shift().rolling(window=rolling_window, min_periods=min_periods).mean())
        )

    # basic matchup features
    df["matchup_diff"] = df["offense_mean_yards"] - df["defense_mean_allowed"]
    df["matchup_ratio"] = (df["offense_mean_yards"] + eps) / (df["defense_mean_allowed"] + eps)
    df["matchup_norm_diff"] = df["matchup_diff"] / (df["offense_mean_yards"] + df["defense_mean_allowed"] + eps)
    df["matchup_interaction"] = df["offense_mean_yards"] * df["defense_mean_allowed"]

    # fill cold starts
    if fill_na_with_global:
        global_avg = df[target].mean()
        df["offense_mean_yards"] = df["offense_mean_yards"].fillna(global_avg)
        df["defense_mean_allowed"] = df["defense_mean_allowed"].fillna(global_avg)
        # recompute dependent signals after fill
        df["matchup_diff"] = df["offense_mean_yards"] - df["defense_mean_allowed"]
        df["matchup_ratio"] = (df["offense_mean_yards"] + eps) / (df["defense_mean_allowed"] + eps)
        df["matchup_norm_diff"] = df["matchup_diff"] / (df["offense_mean_yards"] + df["defense_mean_allowed"] + eps)
        df["matchup_interaction"] = df["offense_mean_yards"] * df["defense_mean_allowed"]

    # compute log matchup ratio with shift to keep arguments positive
    min_val = min(df["offense_mean_yards"].min(skipna=True), df["defense_mean_allowed"].min(skipna=True))
    shift = 0.0
    if pd.notna(min_val) and min_val <= 0:
        shift = -min_val + eps
    df["log_matchup_ratio"] = np.log((df["offense_mean_yards"] + shift + eps) / (df["defense_mean_allowed"] + shift + eps))

    return df

df = add_team_tendency_features(df, rolling_window=None)

toDrop = ['matchup_diff', 'matchup_ratio', "matchup_interaction", 'matchup_norm_diff', 'pff_manZone']
df = df.drop(columns=toDrop)
df


,quarter,down,yardsToGo,possessionTeam,defensiveTeam,absoluteYardlineNumber,offenseFormation,playAction,dropbackType,qbSneak,yardsGained,pff_passCoverage,pff_coverage,leftReceivers,rightReceivers,secondsRemainingInQuarter,offense_mean_yards,defense_mean_allowed,log_matchup_ratio
0,3,1,10,CIN,ATL,31,Empty,False,Traditional,NaN,9,Cover-3,Zone,3,2,114,5.460618,5.460618,0.000000
1,4,1,10,CIN,DAL,18,Empty,False,Traditional,NaN,4,Quarters,Zone,3,2,133,9.000000,5.460618,0.349484
2,4,3,12,HOU,TEN,30,Shotgun,False,Traditional,NaN,6,Quarters,Zone,2,2,120,5.460618,5.460618,0.000000
3,1,2,10,KC,TEN,33,Shotgun,False,Traditional,NaN,4,Quarters,Zone,2,2,568,5.460618,6.000000,-0.061802
4,3,2,8,BAL,TB,37,Pistol,True,Run,False,-1,Cover-1,Man,3,1,136,5.460618,5.460618,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16119,2,3,4,JAX,LV,79,Shotgun,False,Traditional,NaN,0,Cover-2,Zone,2,2,769,5.905904,5.695740,0.023881
16120,4,1,10,MIN,ARI,35,Shotgun,False,Traditional,NaN,0,Cover-3,Zone,2,2,752,5.217842,5.674157,-0.054040
16121,3,1,10,KC,IND,97,Shotgun,False,Scramble,False,10,Quarters,Zone,2,2,599,6.258555,5.062977,0.138265
16122,1,1,10,DAL,CIN,57,Singleback,True,Traditional,NaN,0,Cover-3,Zone,2,2,45,5.446903,5.380863,0.007849


In [147]:
# normalize columns just in case
df.columns = df.columns.str.strip()

# ----- pairwise filtering: keep only rows where both
# (offenseFormation, pff_coverage) and (offenseFormation, pff_passCoverage)
# have >=20 occurrences -----
cnt_off_cov = (
    df.groupby(["offenseFormation", "pff_coverage"])
      .size()
      .rename("cnt_off_cov")
      .reset_index()
)
cnt_off_passcov = (
    df.groupby(["offenseFormation", "pff_passCoverage"])
      .size()
      .rename("cnt_off_passcov")
      .reset_index()
)

# merge counts back
df = df.merge(cnt_off_cov, on=["offenseFormation", "pff_coverage"], how="left")
df = df.merge(cnt_off_passcov, on=["offenseFormation", "pff_passCoverage"], how="left")

# filter out low-frequency combos (<20)
df = df[(df["cnt_off_cov"] >= 20) & (df["cnt_off_passcov"] >= 20)].copy()

# drop the helper count columns
df.drop(columns=["cnt_off_cov", "cnt_off_passcov"], inplace=True)
df

,quarter,down,yardsToGo,possessionTeam,defensiveTeam,absoluteYardlineNumber,offenseFormation,playAction,dropbackType,qbSneak,yardsGained,pff_passCoverage,pff_coverage,leftReceivers,rightReceivers,secondsRemainingInQuarter,offense_mean_yards,defense_mean_allowed,log_matchup_ratio
0,3,1,10,CIN,ATL,31,Empty,False,Traditional,NaN,9,Cover-3,Zone,3,2,114,5.460618,5.460618,0.000000
1,4,1,10,CIN,DAL,18,Empty,False,Traditional,NaN,4,Quarters,Zone,3,2,133,9.000000,5.460618,0.349484
2,4,3,12,HOU,TEN,30,Shotgun,False,Traditional,NaN,6,Quarters,Zone,2,2,120,5.460618,5.460618,0.000000
3,1,2,10,KC,TEN,33,Shotgun,False,Traditional,NaN,4,Quarters,Zone,2,2,568,5.460618,6.000000,-0.061802
4,3,2,8,BAL,TB,37,Pistol,True,Run,False,-1,Cover-1,Man,3,1,136,5.460618,5.460618,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15735,2,3,4,JAX,LV,79,Shotgun,False,Traditional,NaN,0,Cover-2,Zone,2,2,769,5.905904,5.695740,0.023881
15736,4,1,10,MIN,ARI,35,Shotgun,False,Traditional,NaN,0,Cover-3,Zone,2,2,752,5.217842,5.674157,-0.054040
15737,3,1,10,KC,IND,97,Shotgun,False,Scramble,False,10,Quarters,Zone,2,2,599,6.258555,5.062977,0.138265
15738,1,1,10,DAL,CIN,57,Singleback,True,Traditional,NaN,0,Cover-3,Zone,2,2,45,5.446903,5.380863,0.007849


In [148]:
df.to_csv('/Users/Banjo/OneDrive/Desktop/MIDS/Summer 2025/DataSci 207/Final Project/w207FinalProject/data/processed/processed_plays_temp.csv', index=False)

1.

2. Binning / coarsening of key continuous pre-snap scalars
You likely haven’t yet done these (unless in other code):

yardsToGo → bins like Short (≤4), Medium (5–9), Long (≥10) (plus keep the raw value if useful).

absoluteYardlineNumber → zones: Own Red Zone, Own 40, Midfield, Opp 40, Opp Red Zone.

secondsRemainingInQuarter (derived from gameClock) → either raw or coarse buckets (Early, Mid, Late, Final 30s).
Binned versions reduce noise and let embeddings/one-hots capture nonlinearity more cleanly.

3. Interaction / crossed features (explicit)
Deep nets can learn interactions, but giving them obvious ones accelerates convergence:

Down × Distance bin (e.g., combine down with yardsToGo_bin).

Formation × Dropback type (you’ve simplified both; cross them to give the model compound context).

Team tendency matchup × field position zone.

Situation context like (quarter, time remaining) × distance.

These can be implemented as learned “feature crosses” (e.g., concatenated embeddings) or explicit engineered features.

4. Embeddings for remaining categoricals
Instead of one-hot or naive encoding, feed each categorical (including the new binned ones and simplified ones) through trainable embedding layers in your network:

Examples: coverage_simple, dropback_simple, offenseFormation_simple, down, yardsToGo_bin, yardline_zone, team IDs.

Embedding dimension heuristics: ~min(50, round(log2(cardinality)+1)) or proportional to sqrt(cardinality).

This keeps dimensionality low and lets the model learn similarity structure.

5. Target / frequency encoding for coarse high-cardinality categories (if any remain)
If you still have any categorical with many levels (e.g., maybe some residual “other” subtypes), replace their one-hot with:

Target encoding: mean yardsGained per category (with smoothing and cross-fold leakage control).

Frequency encoding: prevalence of each level.
These give dense numeric signal instead of sparse noise.

6. Numeric feature scaling and outlier handling
Standardize or robust-scale all continuous inputs (e.g., tendency differences, raw yardsToGo if used, time).

Cap or transform extreme outliers (e.g., quantile clip or use log/box-cox if distributions are heavy-tailed).
Unnormalized or wild scales slow learning and hurt MAE.

7. Dimensionality reduction on any remaining dense/high-cardinality blocks
If after simplification you still have a block of correlated numeric summaries (e.g., if you later add more pre-snap spatial summaries), compress them:

Autoencoder bottleneck to learn 5–10 latent features.

Or TruncatedSVD / PCA if they’re already vectorized.
This reduces noise and redundancy.

8. Filter/Prune trivial or redundant features
Low-variance filter: drop features with near-zero variance.

Correlation / redundancy check: remove one of highly collinear numeric pairs (or let a small L1 penalty push one toward zero).

9. Temporal/context summarization (if applicable)
If you have access to preceding plays:

Encode short-term “momentum” features like offense’s average yards last 3 plays, defensive stops, turnover tendency, etc.

Keep these summaries low-dimensional (mean, trend) rather than full sequences to avoid complexity explosion.

10. Regularization & architecture considerations to complement simplification
(Not strictly “data” but tightly coupled)

Use dropout / layer norm / weight decay to prevent the net from overfitting residual noise.

Consider a wide-and-deep architecture: wide part for engineered crosses (e.g., team matchup × yardline zone), deep part for embedded raw inputs.

Early stopping based on validation MAE.

11. Calibration of training regime
Stratified validation: ensure splits preserve distribution over key axes (e.g., down-distance, teams) to get reliable MAE estimates.

Sample weighting: if certain situations are overrepresented but less important (e.g., very long yardage), you can downweight them to focus the loss where it matters for overall MAE.

12. Optional augmentation / noise injection
If data is sparse in some regimes, small Gaussian jitter on continuous inputs (within realistic bounds) during training can improve generalization.

Next Concrete Steps (priority order)
Compute and add team tendency and matchup features.

Derive and bin yardsToGo, absoluteYardlineNumber, and gameClock.

Build embeddings for all simplified categoricals and binned features.

Add key interaction crosses (e.g., down×distance, formation×dropback).

Standardize numerics and handle outliers.

Retrain with a validation split, monitor MAE.

If still high, apply target encoding for any remaining categorical leakage and consider a small autoencoder for any correlated numeric summaries.
